# HCDE notebook A: train model v2 → evaluate v2 → export

Model v2 = the v1 training data (reviewed by hand) **plus** the public-benchmark data (MBPP, HumanEval, CodeAlpaca-20k, DS-1000, FinQA, ConvFinQA; automatic quality gate) **plus** the targeted augmentation (SHARED, PROBE, pressure). All of it was prepared on the Mac with `prepare_v2_data.sh` and pushed to GitHub, so this notebook needs no teacher calls, only the judge. About 6 hours.

**Settings (right panel):** Accelerator **GPU T4 x2** · Internet **on**.

**Add-ons → Secrets** (add each and **tick it** for this notebook):

| Secret | Value |
| --- | --- |
| `TEACHER_BASE_URL` | `https://api.groq.com/openai/v1` |
| `TEACHER_API_KEY` | your Groq key(s), comma-separated |
| `TEACHER_MODEL` | `openai/gpt-oss-120b` |
| `JUDGE_MODEL` | `qwen/qwen3.8-27b` (same as in your `.env`) |
| `JUDGE_EXTRA` | `{"reasoning_effort": "none"}` |
| `JUDGE_MAX_TOKENS` | `600` |
| `GITHUB_TOKEN` | only if the AnchorAI repo is private |

**How to run:** **Save Version → Save & Run All (Commit)**. It keeps running with your laptop closed.
When it finishes, open the version → **Output** and download:

* `hcde_v2/hcde-v2-results.zip`: adapter, evaluation results, logs
* `hcde_v2/gguf/hcde-q4_k_m.gguf` + `hcde_v2/gguf/Modelfile`: the model for Ollama

On the Mac: `ollama create hcde-v2 -f Modelfile` (in the folder with both files). The current `hcde` (v1) stays as it is.

In [ ]:
import os, subprocess, time, glob, json, shutil
WORK = "/kaggle/working"
REPO = f"{WORK}/AnchorAI"
BRANCH = "dev/dulanja"
OUTDIR = f"{WORK}/hcde_v2"          # everything worth keeping ends up here
LOGS = f"{OUTDIR}/logs"
os.makedirs(LOGS, exist_ok=True)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
def secret(name):
    try:
        return _secrets.get_secret(name)
    except Exception:
        return ""

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/DulanjaR/AnchorAI.git" if token else "https://github.com/DulanjaR/AnchorAI.git"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

loaded = [k for k in ["TEACHER_BASE_URL", "TEACHER_API_KEY", "TEACHER_MODEL", "TEACHER_EXTRA",
                      "JUDGE_MODEL", "JUDGE_EXTRA", "JUDGE_MAX_TOKENS", "HF_TOKEN"] if secret(k)]
for k in loaded:
    os.environ[k] = secret(k)
HAVE_API = "TEACHER_BASE_URL" in loaded and "TEACHER_API_KEY" in loaded
print("secrets loaded:", loaded or "none")
if not HAVE_API:
    print("WARNING: no Groq secrets -> no judging (replies are still saved; judge them on the Mac)")

subprocess.run("pip install -q -U peft bitsandbytes accelerate && pip uninstall -q -y torchao", shell=True)
GPUS = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
print("GPUs:", GPUS, "(both are used in parallel)" if GPUS > 1 else "(steps run one after another)")

PY = "python -m components.c4_hcde.training"
DATA = "components/c4_hcde/training/data/dataset"

def start(name, cmds, gpu=None):
    """Run shell commands one after another in the background (a failed command does not stop the
    next one). Output goes to logs/<name>.log."""
    env = dict(os.environ)
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    log = open(f"{LOGS}/{name}.log", "w")
    p = subprocess.Popen(" ; ".join(cmds), shell=True, cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT)
    p.name = name
    print(f"started {name}" + (f" on GPU {gpu}" if gpu is not None else ""))
    return p

def tail(name, n=2):
    try:
        lines = [l.rstrip() for l in open(f"{LOGS}/{name}.log", errors="replace").read().replace("\r", "\n").splitlines() if l.strip()]
        return " | ".join(lines[-n:])[-300:]
    except FileNotFoundError:
        return ""

def wait(procs, every=300):
    """Block until all background jobs finish, printing progress every few minutes."""
    t0 = last = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(10)
        if time.time() - last < every and any(p.poll() is None for p in procs):
            continue
        last = time.time()
        print(f"--- {int((time.time() - t0) / 60)} min")
        for p in procs:
            print(f"  {p.name}: {'running' if p.poll() is None else 'done'} :: {tail(p.name)}", flush=True)
    for p in procs:
        print(f"{p.name}: exit code {p.returncode}  :: {tail(p.name, 3)}")

def run_on_gpus(chains):
    """chains = list of (name, [commands]); one chain per GPU, in parallel when there are 2 GPUs."""
    if GPUS > 1:
        wait([start(n, c, gpu=i % GPUS) for i, (n, c) in enumerate(chains)])
    else:
        for n, c in chains:
            wait([start(n, c, gpu=0)])

In [ ]:
EXTRA = [f for f in [f"{DATA}/public_train.jsonl", f"{DATA}/aug_shared.jsonl", f"{DATA}/aug_probe.jsonl",
                     f"{DATA}/aug_pressure.jsonl"] if os.path.exists(f)]
for f in EXTRA:
    rows = [json.loads(l) for l in open(f) if l.strip()]
    print(f"{f.split('/')[-1]}: {sum(r.get('accepted', True) for r in rows)} accepted of {len(rows)}")
print("public test set:", os.path.exists(f"{DATA}/public_test.jsonl"))
if len(EXTRA) < 4:
    print("WARNING: some v2 data files are missing; did prepare_v2_data.sh finish and was it pushed?")

## 2. Train model v2 (QLoRA, 2 epochs, about 4 hours)
Same settings as v1 (Qwen2.5-7B-Instruct, r 16, alpha 32), so the difference between v1 and v2 is the data.

In [ ]:
RUN = f"{OUTDIR}/run"
if os.path.exists(f"{RUN}/adapter/adapter_config.json"):
    print("adapter already trained, skipping")
else:
    extra = ("--extra-train " + " ".join(EXTRA)) if EXTRA else ""
    wait([start("train", [f"{PY}.train_qlora --base-model Qwen/Qwen2.5-7B-Instruct --out {RUN} --epochs 2 {extra}"], gpu=0)])
try:
    m = json.load(open(f"{RUN}/training_meta.json"))
    print("best eval loss:", m["eval"].get("eval_loss"), "| minutes:", m["train_minutes"])
except Exception as e:
    print("TRAINING DID NOT FINISH, see logs/train.log:", e, "\n", tail("train", 15))

## 3. Evaluate v2 (about 1 hour, both GPUs)
IT test set, transfer set and the public-benchmark test set, each normally and under pressure. Only the tuned model here: the base-model and v1 results come from notebooks B and C (same test files, same settings).

In [ ]:
AD = f"--adapter {RUN}/adapter --conditions tuned_tag --tuned-name tuned_v2"
TR = f"--test {DATA}/transfer_raw.jsonl --elements {DATA}/transfer_elements.json"
PB = f"--test {DATA}/public_test.jsonl --elements {DATA}/public_elements.json"
run_on_gpus([
    ("eval_it", [f"{PY}.evaluate {AD} --out {OUTDIR}/eval",
                 f"{PY}.evaluate {AD} --pressure --out {OUTDIR}/eval-pressure",
                 f"{PY}.evaluate {AD} {PB} --out {OUTDIR}/eval-public"]),
    ("eval_transfer", [f"{PY}.evaluate {AD} {TR} --out {OUTDIR}/eval-transfer",
                       f"{PY}.evaluate {AD} {TR} --pressure --out {OUTDIR}/eval-transfer-pressure",
                       f"{PY}.evaluate {AD} {PB} --pressure --out {OUTDIR}/eval-public-pressure"]),
])

## 4. Judge (SDS) and export to GGUF, in parallel (about 1 hour)

In [ ]:
subprocess.run("apt-get -qq install -y cmake > /dev/null", shell=True)
export = start("export_gguf", [f"{PY}.export_gguf --adapter {RUN}/adapter --out {OUTDIR}/gguf "
                               "--llama-cpp /tmp/llama.cpp --work-dir /tmp/hcde-export"])

if HAVE_API:
    jobs = []
    for d, elements in [(f"{OUTDIR}/eval", f"{DATA}/solution_elements.json"),
                     (f"{OUTDIR}/eval-pressure", f"{DATA}/solution_elements.json"),
                     (f"{OUTDIR}/eval-transfer", f"{DATA}/transfer_elements.json"),
                     (f"{OUTDIR}/eval-transfer-pressure", f"{DATA}/transfer_elements.json"),
                     (f"{OUTDIR}/eval-public", f"{DATA}/public_elements.json"),
                     (f"{OUTDIR}/eval-public-pressure", f"{DATA}/public_elements.json")]:
        if os.path.exists(f"{d}/generations.jsonl"):
            name = "judge_" + d.rstrip("/").split("/")[-1]
            # run twice: the second pass only re-scores rows the first one could not (rate limits)
            cmd = f"{PY}.evaluate --from-generations {d}/generations.jsonl --elements {elements} --out {d} --judge"
            jobs.append(start(name, [cmd, cmd]))
    wait(jobs)
else:
    print("no Groq secrets: replies are saved, judge them later on the Mac with evaluate --from-generations ... --judge")

wait([export])
for d in ["eval", "eval-pressure", "eval-transfer", "eval-transfer-pressure", "eval-public", "eval-public-pressure"]:
    p = f"{OUTDIR}/{d}/summary.md"
    if os.path.exists(p):
        print(f"\n### {d}\n" + open(p).read())

## 5. Package the results

In [ ]:
os.chdir(OUTDIR)
subprocess.run("zip -qr hcde-v2-results.zip run/adapter run/training_meta.json eval eval-pressure "
               "eval-transfer eval-transfer-pressure eval-public eval-public-pressure logs", shell=True)
os.chdir(REPO)
for f in sorted(glob.glob(f"{OUTDIR}/**", recursive=True)):
    if os.path.isfile(f) and (f.endswith((".zip", ".gguf", "Modelfile", "summary.md"))):
        print(f"{os.path.getsize(f) / 1e6:9.1f} MB  {f}")